# EXP19: 赢家/输家交易画像

分析赢家和输家的特征差异，寻找可利用的模式。

In [ ]:
import sys; sys.path.insert(0, '..')
from backtest import DataBundle, run_variant, C12_KWARGS
import pandas as pd
import numpy as np

data = DataBundle.load_default()

LIVE_KWARGS = {
    **C12_KWARGS,
    "intraday_adaptive": True,
    "choppy_threshold": 0.35,
    "kc_only_threshold": 0.60,
    "spread_cost": 0.50,
}

baseline = run_variant(data, "Baseline", **LIVE_KWARGS)
trades = baseline['_trades']
print(f"Total: {len(trades)} trades")

In [ ]:
df = pd.DataFrame([{
    'pnl': t.pnl,
    'win': t.pnl > 0,
    'strategy': t.strategy,
    'direction': t.direction,
    'hour': t.entry_time.hour,
    'dow': t.entry_time.weekday(),
    'bars_held': t.bars_held,
    'exit_reason': t.exit_reason.split(':')[0] if ':' in t.exit_reason else t.exit_reason,
    'month': t.entry_time.month,
    'year': t.entry_time.year,
} for t in trades])

winners = df[df['win']]
losers = df[~df['win']]

## Part 1: 基础统计对比

In [ ]:
print(f"=== Winners vs Losers ===")
print(f"Winners: {len(winners)} ({len(winners)/len(df)*100:.1f}%), avg=${winners['pnl'].mean():.2f}, median=${winners['pnl'].median():.2f}")
print(f"Losers:  {len(losers)} ({len(losers)/len(df)*100:.1f}%), avg=${losers['pnl'].mean():.2f}, median=${losers['pnl'].median():.2f}")
print(f"\nAvg bars: Winners={winners['bars_held'].mean():.1f}, Losers={losers['bars_held'].mean():.1f}")
print(f"Median bars: Winners={winners['bars_held'].median():.0f}, Losers={losers['bars_held'].median():.0f}")
print(f"\nProfit factor: ${winners['pnl'].sum():.0f} / ${abs(losers['pnl'].sum()):.0f} = {winners['pnl'].sum()/abs(losers['pnl'].sum()):.2f}")

In [ ]:
# PnL 分布
print("\n=== PnL Distribution ===")
for pct in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
    print(f"  P{pct:2d}: ${np.percentile(df['pnl'], pct):7.2f}")

# 大赢家和大输家
print(f"\nBig winners (>$10): {(df['pnl'] > 10).sum()} trades, ${df[df['pnl'] > 10]['pnl'].sum():.0f}")
print(f"Big losers (<-$10): {(df['pnl'] < -10).sum()} trades, ${df[df['pnl'] < -10]['pnl'].sum():.0f}")
print(f"Small trades (|pnl|<$2): {(df['pnl'].abs() < 2).sum()} trades, ${df[df['pnl'].abs() < 2]['pnl'].sum():.0f}")

## Part 2: 特征差异

In [ ]:
# 赢家 vs 输家 的入场时间分布
print("=== Entry Hour Distribution ===")
w_hours = winners['hour'].value_counts().sort_index()
l_hours = losers['hour'].value_counts().sort_index()
compare = pd.DataFrame({'winners': w_hours, 'losers': l_hours}).fillna(0)
compare['win_rate'] = compare['winners'] / (compare['winners'] + compare['losers'])
print(compare.round(3))

In [ ]:
# 赢家 vs 输家 的 exit reason 分布
print("\n=== Exit Reason by Win/Loss ===")
cross = pd.crosstab(df['exit_reason'], df['win'], margins=True)
cross.columns = ['Loss', 'Win', 'Total']
cross['WR'] = cross['Win'] / cross['Total']
print(cross.round(3))

print("\n=== Direction by Win/Loss ===")
cross2 = pd.crosstab(df['direction'], df['win'], margins=True)
cross2.columns = ['Loss', 'Win', 'Total']
cross2['WR'] = cross2['Win'] / cross2['Total']
print(cross2.round(3))

In [ ]:
# 连胜/连败分析
streaks = []
current = 0
current_type = None
for _, row in df.iterrows():
    w = row['win']
    if w == current_type:
        current += 1
    else:
        if current_type is not None:
            streaks.append(('W' if current_type else 'L', current))
        current = 1
        current_type = w
streaks.append(('W' if current_type else 'L', current))

win_streaks = [s[1] for s in streaks if s[0] == 'W']
loss_streaks = [s[1] for s in streaks if s[0] == 'L']

print(f"\n=== Streaks ===")
print(f"Win streaks: max={max(win_streaks)}, avg={np.mean(win_streaks):.1f}, count={len(win_streaks)}")
print(f"Loss streaks: max={max(loss_streaks)}, avg={np.mean(loss_streaks):.1f}, count={len(loss_streaks)}")
print(f"Loss streaks >= 5: {sum(1 for s in loss_streaks if s >= 5)}")
print(f"Loss streaks >= 10: {sum(1 for s in loss_streaks if s >= 10)}")